In [ ]:
from pathlib import Path

import myo
import myoktros
import numpy as np
import pandas as pd
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
DATA_PATH = Path('.') / "data"
ARM_DOMINANCE = "right",
EMG_MODE = myo.types.EMGMode.SEND_FILT
N_SAMPLES = 5

In [ ]:
# load data
df = myoktros.GestureModel.read_data(
    DATA_PATH,
    ARM_DOMINANCE,
    EMG_MODE,
    N_SAMPLES,
)
df

In [ ]:
features = df.copy()

# reserve 10% samples for validation
X_val = features.groupby('gesture').apply(lambda x: x.sample(frac=0.1)).reset_index(drop=True)

# split the data into features and labels
labels = features.pop('gesture')
y_val = X_val.pop('gesture')

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.33, random_state=42)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

In [ ]:
# training
max_epochs = 1000

accs = []
val_accs = []
losses = []
val_losses = []

best_model = None
best_val_loss = 0

# model + compilation
model = tf.keras.Sequential([
    tf.keras.layers.Dense(200, activation = tf.nn.sigmoid, input_shape = (X_train.shape[1],)),
    tf.keras.layers.Dense(100, activation = tf.nn.sigmoid),
    tf.keras.layers.Dense(50, activation = tf.nn.sigmoid),
    tf.keras.layers.Dense(len(myoktros.gesture.Gesture), activation = tf.nn.sigmoid)
])
model.compile(optimizer = tf.keras.optimizers.Adam(),
              loss = 'sparse_categorical_crossentropy',
              metrics = ['accuracy'])

for i in range(0, max_epochs):
    print('EPOCH:', i + 1, '/', max_epochs)

    h = model.fit(
        x=X_train,
        y=y_train,
        validation_data=(X_val, y_val),
    )

    accs.append(h.history.get('accuracy')[0])
    val_accs.append(h.history.get('val_accuracy')[0])
    losses.append(h.history.get('loss')[0])
    val_losses.append(h.history.get('val_loss')[0])

    # check if model is better
    if best_model == None:
        best_model = model
        best_val_loss = val_losses[-1]
    elif best_val_loss > val_losses[-1]:
        best_model = model
        best_val_loss = val_losses[-1]

    # quit if critera is met
    if val_losses[-1] / best_val_loss > 1.25:
        break

model = best_model
p = Path('.') / "assets" / f"keras-{EMG_MODE.name}-{N_SAMPLES}-samples-model"
model.save(p.absolute())

In [ ]:
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)

# normalize="pred": 
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# labels are gestures
legend = [g.name for g in myoktros.Gesture]

plt.imshow(cm)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.yticks(np.arange(len(legend)), legend)
plt.xticks(np.arange(len(legend)), legend, rotation='vertical')
plt.colorbar()